# 01 — Dataset inventory

This notebook performs the first structural inspection of the raw datasets used in the pan-cancer epigenetic resistance project.

The goal is to verify:

- file availability
- file size
- table dimensions
- column structure
- identifier columns
- missingness
- duplicated keys
- basic compatibility between DepMap and GDSC inputs

Raw data are treated as read-only.

In [1]:
# !/usr/bin/env python3

from pathlib import Path
import pandas as pd

In [2]:
# Define project root and data directories

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "data" / "raw").exists() and (path / "README.md").exists():
            return path
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = find_project_root(Path.cwd())

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DEPMAP_DIR = DATA_RAW / "depmap"
GDSC_DIR = DATA_RAW / "gdsc"

FILES = {
    "depmap_model": DEPMAP_DIR / "Model.csv",
    "depmap_omics_profiles": DEPMAP_DIR / "OmicsProfiles.csv",
    "depmap_expression": DEPMAP_DIR / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv",
    "depmap_mutations": DEPMAP_DIR / "OmicsSomaticMutations.csv",
    "gdsc_ic50": GDSC_DIR / "GDSC2_fitted_dose_response_27Oct23.xlsx",
}

PROJECT_ROOT

WindowsPath('C:/Users/paula/OneDrive/Documentos/Proyectos/pancancer-epigenetics')

In [3]:
# Check if files exist and print their sizes
for name, path in FILES.items():
    size_mb = path.stat().st_size / 1e6 if path.exists() else None
    print(f"{name:25} | exists={path.exists()} | size_mb={size_mb}")

depmap_model              | exists=True | size_mb=0.699474
depmap_omics_profiles     | exists=True | size_mb=0.663971
depmap_expression         | exists=True | size_mb=543.026959
depmap_mutations          | exists=True | size_mb=595.799492
gdsc_ic50                 | exists=True | size_mb=21.330376


In [4]:
# Preview the first few rows of each file and collect inventory information
inventory_rows = []

for name, path in FILES.items():

    if path.suffix == ".csv":
        df = pd.read_csv(path, nrows=5)

    elif path.suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(path, nrows=5)

    else:
        continue

    inventory_rows.append({
        "dataset": name,
        "rows_preview": df.shape[0],
        "columns": df.shape[1],
        "column_names_preview": list(df.columns[:10]),
    })

inventory_df = pd.DataFrame(inventory_rows)

inventory_df

,dataset,rows_preview,columns,column_names_preview
0,depmap_model,5,49,"[ModelID, PatientID, CellLineName, StrippedCel..."
1,depmap_omics_profiles,5,18,"[ModelID, IsDefaultEntryForModel, StrippedCell..."
2,depmap_expression,5,19221,"[Unnamed: 0, SequencingID, ModelID, IsDefaultE..."
3,depmap_mutations,5,69,"[Unnamed: 0, SequencingID, ModelID, ModelCondi..."
4,gdsc_ic50,5,19,"[DATASET, NLME_RESULT_ID, NLME_CURVE_ID, COSMI..."


In [ ]:
# Print column names for each file, handling cases with many columns
for name, path in FILES.items():
    print("=" * 80)
    print(name)
    
    if path.suffix == ".csv":
        df_preview = pd.read_csv(path, nrows=5)
    else:
        df_preview = pd.read_excel(path, nrows=5)
    
    print(f"Columns: {df_preview.shape[1]}")
    
    if df_preview.shape[1] <= 80:
        print(list(df_preview.columns))
    else:
        print("First 15 columns:")
        print(list(df_preview.columns[:15]))
        print("Last 15 columns:")
        print(list(df_preview.columns[-15:]))

depmap_model
Columns: 49
['ModelID', 'PatientID', 'CellLineName', 'StrippedCellLineName', 'DepmapModelType', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype', 'OncotreeCode', 'PatientSubtypeFeatures', 'RRID', 'Age', 'AgeCategory', 'Sex', 'PatientRace', 'PrimaryOrMetastasis', 'SampleCollectionSite', 'SourceType', 'SourceDetail', 'CatalogNumber', 'ModelType', 'TissueOrigin', 'ModelDerivationMaterial', 'ModelTreatment', 'PatientTreatmentStatus', 'PatientTreatmentType', 'PatientTreatmentDetails', 'Stage', 'StagingSystem', 'PatientTumorGrade', 'PatientTreatmentResponse', 'GrowthPattern', 'OnboardedMedia', 'FormulationID', 'SerumFreeMedia', 'PlateCoating', 'EngineeredModel', 'EngineeredModelDetails', 'CulturedResistanceDrug', 'PublicComments', 'CCLEName', 'HCMIID', 'PediatricModelType', 'ModelAvailableInDbgap', 'ModelSubtypeFeatures', 'WTSIMasterCellID', 'SangerModelID', 'COSMICID', 'ModelIDAlias']
depmap_omics_profiles
Columns: 18
['ModelID', 'IsDefaultEntryForModel', 'Strippe

In [7]:
# Check for key columns and their properties
key_checks = []

checks = {
    "depmap_model": ("csv", FILES["depmap_model"], ["ModelID", "SangerModelID", "COSMICID", "CellLineName"]),
    "depmap_omics_profiles": ("csv", FILES["depmap_omics_profiles"], ["ModelID", "ProfileID", "SequencingID"]),
    "depmap_expression": ("csv", FILES["depmap_expression"], ["ModelID", "SequencingID"]),
    "depmap_mutations": ("csv", FILES["depmap_mutations"], ["ModelID", "SequencingID", "HugoSymbol"]),
    "gdsc_ic50": ("excel", FILES["gdsc_ic50"], ["SANGER_MODEL_ID", "COSMIC_ID", "CELL_LINE_NAME", "DRUG_NAME"]),
}

for dataset, (kind, path, cols) in checks.items():
    if kind == "csv":
        df = pd.read_csv(path, usecols=lambda c: c in cols)
    else:
        df = pd.read_excel(path, usecols=lambda c: c in cols)

    for col in cols:
        if col in df.columns:
            key_checks.append({
                "dataset": dataset,
                "column": col,
                "rows": len(df),
                "missing": df[col].isna().sum(),
                "unique": df[col].nunique(dropna=True),
                "duplicated_values": df[col].duplicated().sum(),
            })

key_checks_df = pd.DataFrame(key_checks)
key_checks_df

,dataset,column,rows,missing,unique,duplicated_values
0,depmap_model,ModelID,2132,0,2132,0
1,depmap_model,SangerModelID,2132,914,1217,914
2,depmap_model,COSMICID,2132,1155,977,1154
3,depmap_model,CellLineName,2132,0,2132,0
4,depmap_omics_profiles,ModelID,4775,0,2031,2744
5,depmap_omics_profiles,ProfileID,4775,0,4723,52
6,depmap_omics_profiles,SequencingID,4775,0,4775,0
7,depmap_expression,ModelID,1754,0,1699,55
8,depmap_expression,SequencingID,1754,0,1754,0
9,depmap_mutations,ModelID,1177454,0,1955,1175499


In [9]:
# Focused exploration of the Omics Profiles dataset
omics_profiles = pd.read_csv(FILES["depmap_omics_profiles"])

omics_profiles[
    [
        "ModelID",
        "ProfileID",
        "SequencingID",
        "DataType",
        "IsDefaultEntryForModel",
        "IsDefaultEntryForMC",
    ]
].head(20)

,ModelID,ProfileID,SequencingID,DataType,IsDefaultEntryForModel,IsDefaultEntryForMC
0,ACH-000001,PR-vFBZ2h,CDS-VqxBGH,rna,Yes,Yes
1,ACH-000001,PR-sbeUkh,CDS-1pLoxn,wgs,Yes,Yes
2,ACH-000001,PR-99siqG,CDS-C3gZIu,wes,No,Yes
3,ACH-000013,PR-gXHYvm,CDS-qqF7T8,rna,Yes,Yes
4,ACH-000013,PR-yFRI6x,CDS-FvwPIb,wes,No,No
5,ACH-000013,PR-s6xfW4,CDS-w4IdeE,wgs,Yes,Yes
6,ACH-000103,PR-kpBxri,CDS-kpzIIP,rna,Yes,Yes
7,ACH-000103,PR-L5V3oJ,CDS-Nw4mGG,wes,Yes,Yes
8,ACH-000116,PR-GxfIvz,CDS-Dc4piF,rna,Yes,Yes
9,ACH-000116,PR-EjK0zK,CDS-bGOzNq,wgs,Yes,Yes


In [ ]:
# Summarize the Omics Profiles dataset by DataType and IsDefaultEntryForModel
omics_profiles = pd.read_csv(FILES["depmap_omics_profiles"])

summary = (
    omics_profiles
    .groupby(["DataType", "IsDefaultEntryForModel"])
    .agg(
        rows=("ModelID", "size"),
        unique_models=("ModelID", "nunique"),
        unique_sequencing=("SequencingID", "nunique"),
    )
    .reset_index()
)

summary

,DataType,IsDefaultEntryForModel,rows,unique_models,unique_sequencing
0,rna,No,55,53,55
1,rna,Yes,1699,1699,1699
2,wes,No,1052,931,1052
3,wes,Yes,860,860,860
4,wgs,No,14,14,14
5,wgs,Yes,1095,1095,1095


In [12]:
# Explore the overlap of Sanger Model IDs between DepMap and GDSC datasets
model_df = pd.read_csv(FILES["depmap_model"])

gdsc_df = pd.read_excel(FILES["gdsc_ic50"])

depmap_sanger_ids = (
    model_df["SangerModelID"]
    .dropna()
    .astype(str)
    .unique()
)

gdsc_sanger_ids = (
    gdsc_df["SANGER_MODEL_ID"]
    .dropna()
    .astype(str)
    .unique()
)

shared_ids = set(depmap_sanger_ids) & set(gdsc_sanger_ids)

print(f"DepMap Sanger IDs : {len(depmap_sanger_ids)}")
print(f"GDSC Sanger IDs   : {len(gdsc_sanger_ids)}")
print(f"Shared IDs        : {len(shared_ids)}")

DepMap Sanger IDs : 1217
GDSC Sanger IDs   : 969
Shared IDs        : 967
